# P10.6-AI — Notebook 59: entrenamiento foraminal Sagittal T1

Entrena un clasificador **2.5D multiclase** para estrechamiento foraminal neural izquierdo y derecho usando exclusivamente `train_manifest.csv` y `validation_manifest.csv` del Notebook 58.

El modelo comparte el backbone entre ambos lados e incorpora embeddings explícitos de **lado** y **nivel lumbar**. El `internal_test` permanece sellado para el Notebook 60.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


## Recursos y alcance

- Runtime recomendado: **Google Colab con GPU T4 o superior**.
- Autorizar Google Drive.
- No requiere token de GitHub.
- El token de Kaggle se solicita únicamente si las series Sagittal T1 necesarias no están disponibles en el runtime o en Drive.
- El notebook puede reanudarse: los crops `.npy` ya generados y el checkpoint se reutilizan.
- No abre ni carga `internal_test_manifest.csv`.


In [ ]:
# 1) Dependencias mínimas
from __future__ import annotations
import importlib.util
import subprocess
import sys

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "kaggle": "kaggle",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *missing,
    ])
print({"installedNow": missing})


In [ ]:
# 2) GPU y Google Drive
import getpass
import json
import os
from pathlib import Path

import torch
from google.colab import drive  # type: ignore

if not torch.cuda.is_available():
    raise RuntimeError(
        "Seleccioná GPU T4 o superior: Entorno de ejecución > Cambiar tipo de entorno."
    )

print({
    "gpu": torch.cuda.get_device_name(0),
    "gpuMemoryGiB": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2,
    ),
    "torch": torch.__version__,
})
drive.mount("/content/drive", force_remount=False)


In [ ]:
# 3) Clonar o actualizar la rama e importar el pipeline
REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git",
        "clone",
        "--branch",
        REPO_REF,
        "--single-branch",
        REPO_URL,
        str(REPO_ROOT),
    ])
else:
    subprocess.check_call(
        ["git", "fetch", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "checkout", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "pull", "--ff-only", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()

sys.path.insert(0, str(REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_foraminal_training import (
    TrainConfig,
    build_cache,
    download_selected_series,
    find_data_root,
    load_manifests,
    prepare_samples,
    train_model,
)

print({"repoRef": REPO_REF, "repoSha": REPO_SHA})


In [ ]:
# 4) Rutas y configuración
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
RESULTS_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
SPLIT_ROOT = RESULTS_ROOT / "notebook58_foraminal_split"
RUN_ROOT = RESULTS_ROOT / "notebook59_foraminal_training"

MODEL_ROOT = (
    PFI_ROOT
    / "models"
    / "P10_6_rsna_findings"
    / "foraminal_sagittal_t1_2p5d"
)
CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"

LOCAL_DATA_ROOT = Path("/content/RSNA_LUMBAR_DISC")
DRIVE_DATA_ROOT = PFI_ROOT / "data" / "RSNA_LUMBAR_DISC"
CACHE_ROOT = Path("/content/rsna_foraminal_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"

CFG = TrainConfig(
    seed=2026,
    image_size=224,
    crop_size=256,
    batch_size=32,
    num_workers=2,
    max_epochs=15,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    model_name="efficientnet_b0",
    pretrained=True,
)

for path in (RUN_ROOT, MODEL_ROOT, CHECKPOINT_ROOT, CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(CFG)
print({
    "splitRoot": str(SPLIT_ROOT),
    "runRoot": str(RUN_ROOT),
    "checkpointRoot": str(CHECKPOINT_ROOT),
    "cacheRoot": str(CACHE_ROOT),
})


## Carga y auditoría de datos

La función siguiente verifica hashes, aprobación del Notebook 58, separación por estudio y presencia de las tres clases. Solo lee los manifests de entrenamiento y validación; del internal test únicamente comprueba que el archivo sellado exista.


In [ ]:
# 5) Cargar train y validation sin acceder al internal test
train_manifest, validation_manifest, split_summary, manifest_hashes = (
    load_manifests(SPLIT_ROOT)
)

distribution = (
    train_manifest.groupby(["side", "level", "severity"])
    .size()
    .rename("train_rows")
    .reset_index()
)
validation_distribution = (
    validation_manifest.groupby(["side", "level", "severity"])
    .size()
    .rename("validation_rows")
    .reset_index()
)

print({
    "trainRows": len(train_manifest),
    "trainStudies": train_manifest["study_id"].nunique(),
    "validationRows": len(validation_manifest),
    "validationStudies": validation_manifest["study_id"].nunique(),
    "trainClassCounts": train_manifest["severity"].value_counts().to_dict(),
    "validationClassCounts": validation_manifest["severity"].value_counts().to_dict(),
    "internalTestAccessed": False,
    "officialTestAccessed": False,
})
display(distribution)
display(validation_distribution)


In [ ]:
# 6) Resolver las series Sagittal T1 requeridas
DATA_ROOT, data_audits = find_data_root(
    [LOCAL_DATA_ROOT, DRIVE_DATA_ROOT],
    train_manifest,
    validation_manifest,
)

print({
    "audits": [
        {
            "root": audit.root,
            "complete": audit.complete,
            "requiredSeries": audit.required_series,
            "missingSeries": audit.missing_series,
            "missingExamples": list(audit.missing_examples[:5]),
        }
        for audit in data_audits
    ]
})

if DATA_ROOT is None:
    token = getpass.getpass(
        "Pegá tu KAGGLE_API_TOKEN para descargar solo las series requeridas: "
    ).strip()
    DATA_ROOT = download_selected_series(
        train_manifest,
        validation_manifest,
        LOCAL_DATA_ROOT,
        COMPETITION,
        token,
    )
    token = ""
    os.environ.pop("KAGGLE_API_TOKEN", None)

print({
    "dataRoot": str(DATA_ROOT),
    "sequence": "Sagittal T1",
    "internalTestAccessed": False,
})


## Preparación 2.5D

Cada muestra usa tres cortes sagitales adyacentes (`centro-1`, `centro`, `centro+1`) alrededor de la coordenada foraminal. Se recorta una región de 256×256 y se redimensiona a 224×224. No se aplica espejo horizontal porque invertiría el eje anteroposterior.


In [ ]:
# 7) Preparar muestras y construir/reutilizar caché local
train_samples = prepare_samples(train_manifest, DATA_ROOT, "train")
validation_samples = prepare_samples(
    validation_manifest,
    DATA_ROOT,
    "validation",
)

train_cache_audit = build_cache(
    train_samples,
    CACHE_ROOT,
    "train",
    CFG,
)
validation_cache_audit = build_cache(
    validation_samples,
    CACHE_ROOT,
    "validation",
    CFG,
)

print({
    "trainCache": train_cache_audit,
    "validationCache": validation_cache_audit,
    "internalTestAccessed": False,
})


## Entrenamiento

El muestreo pondera simultáneamente la clase y el estrato `lado × nivel × severidad`. La selección del checkpoint combina macro F1, balanced accuracy, recall de `Severe` y recall de `Moderate`. El entrenamiento usa AMP, AdamW, reducción de learning rate y early stopping.


In [ ]:
# 8) Entrenar y seleccionar el checkpoint usando validation
torch.cuda.empty_cache()

training_summary = train_model(
    train_samples,
    validation_samples,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
    RUN_ROOT,
    manifest_hashes,
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
    config=CFG,
)

print(json.dumps({
    "status": training_summary["status"],
    "approved": training_summary["approved"],
    "nextNotebook": training_summary["nextNotebook"],
    "bestEpoch": training_summary["bestEpoch"],
    "bestSelectionScore": training_summary["bestSelectionScore"],
    "validationMetrics": training_summary["validationMetrics"],
    "processGates": training_summary["processGates"],
    "metricGates": training_summary["metricGates"],
    "checkpoint": training_summary["checkpoint"],
    "internalTestAccessed": False,
}, indent=2, ensure_ascii=False))


In [ ]:
# 9) Gate final y evidencia generada
required_outputs = [
    "training_history.csv",
    "validation_predictions.csv",
    "validation_metrics_by_group.csv",
    "sampling_audit.json",
    "model_card.md",
    "training_summary.json",
]
missing_outputs = [
    name for name in required_outputs
    if not (RUN_ROOT / name).is_file()
]

if missing_outputs:
    raise RuntimeError(f"Faltan outputs del Notebook 59: {missing_outputs}")
if not (CHECKPOINT_ROOT / "best_checkpoint.pt").is_file():
    raise RuntimeError("No se generó best_checkpoint.pt.")

print({
    "status": training_summary["status"],
    "outputs": required_outputs,
    "bestCheckpoint": str(CHECKPOINT_ROOT / "best_checkpoint.pt"),
    "internalTestSealedUntilNotebook60": True,
    "officialTestAccessed": False,
    "humanReviewRequired": True,
    "notClinicalDiagnosis": True,
})

if not training_summary["approved"]:
    raise RuntimeError(
        "El entrenamiento terminó, pero no superó todos los gates de validación. "
        "Revisar training_summary.json antes del Notebook 60."
    )


## Conclusión esperada

Una ejecución aprobada termina con `APPROVED_FOR_NOTEBOOK_60`. El checkpoint seleccionado y sus métricas quedan congelados en Drive. Recién el Notebook 60 podrá abrir el `internal_test_manifest.csv` para realizar una única evaluación final y decidir la exportación del modelo.
